# Loading Data


In [17]:
import pandas as pd
import ast
import numpy as np

In [63]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/dnb-handle-data")

print("Path to dataset files:", path)

100%|██████████| 319k/319k [00:00<00:00, 658kB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/dnb-handle-data/versions/2


In [18]:
mbm_bat = pd.read_csv(r'D:\ED\FIFS_IPL\data\mbm\cricsheet_ipl_mbm_bat.csv')
playing_xi = pd.read_csv(r'D:\ED\FIFS_IPL\data\register_data\playing_xi.csv')
player_info = pd.read_csv(r'D:\ED\FIFS_IPL\data\register_data\player_details.csv')

# Handling DNB Values

In [19]:
# Update team names in team1_name and team2_name columns
playing_xi['team1_name'] = playing_xi['team1_name'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})
playing_xi['team2_name'] = playing_xi['team2_name'].replace({'Delhi Daredevils': 'Delhi Capitals',
                                                               'Kings XI Punjab': 'Punjab Kings',
                                                            'Rising Pune Supergiants': 'Rising Pune Supergiant'})

In [7]:
mbm_bat['batter_id'] = mbm_bat['batter_id'].astype(str)

def parse_playing_xi(x):
    try:
        return ast.literal_eval(x)
    except Exception:
        return []

# Apply the conversion on both team1 and team2 playing XI columns
playing_xi['team1_playing_xi'] = playing_xi['team1_playing_xi'].apply(parse_playing_xi)
playing_xi['team2_playing_xi'] = playing_xi['team2_playing_xi'].apply(parse_playing_xi)

# Define a helper function that returns the longest non-null string among the three names
def get_longest_name(row):
    names = [row['cricsheet_name'], row['unique_name'], row['cricbuzz_name']]
    # Filter out any missing values
    names = [name for name in names if pd.notnull(name)]
    return max(names, key=len) if names else ''

# Create a mapping dictionary where the key is the player_id (converted to string)
# and the value is the longest name obtained by applying the function above
player_map = dict(zip(player_info['key_cricinfo'].astype(str),
                      player_info.apply(get_longest_name, axis=1)))

In [8]:
dnb_rows = []

# Group mbm_bat by match_id and innings (since a match can have multiple innings)
for (match_id, inns), group in mbm_bat.groupby(['match_id', 'inns']):
    # Get common details from the group (assumed to be the same for all rows in the group)
    date = group['date'].iloc[0]
    bat_team = group['bat_team'].iloc[0]
    bowl_team = group['bowl_team'].iloc[0]
    venue = group['venue'].iloc[0]

    # Determine the set of batter IDs (as strings) who batted in this innings
    batted_ids = set(group['batter_id'].unique())

    # Lookup the corresponding playing_xi row for this match_id
    xi_row = playing_xi[playing_xi['match_id'] == match_id]
    if xi_row.empty:
        print(f"Warning: Match ID {match_id} not found in playing XI data.")
        continue

    row = xi_row.iloc[0]
    # Determine which playing XI list corresponds to the batting team
    if row['team1_name'] == bat_team:
        full_xi = row['team1_playing_xi']
    elif row['team2_name'] == bat_team:
        full_xi = row['team2_playing_xi']
    else:
        print(f"Warning: Batting team {bat_team} not found in playing XI teams for match {match_id}.")
        continue

    # Convert all player IDs in the playing XI list to strings
    full_xi = [str(pid) for pid in full_xi]

    # Identify players in the playing XI who did not bat (i.e. DNB)
    dnb_ids = [pid for pid in full_xi if pid not in batted_ids]

    # For each DNB player, create a new row with default values
    for pid in dnb_ids:
        batter_name = player_map.get(pid, "")
        new_row = {
            'date': date,
            'match_id': match_id,
            'bat_team': bat_team,
            'bowl_team': bowl_team,
            'inns': inns,
            'venue': venue,
            'batter_id': pid,
            'batter': batter_name,
            'runs': 0,
            'balls': 0,
            'fours': 0,
            'sixes': 0,
            'sr': 0,
            'isOut': "DNB",
            'wicketType': "DNB",
            'bowler_id': "",
            'bowler': ""
        }
        dnb_rows.append(new_row)

In [9]:
dnb_df = pd.DataFrame(dnb_rows)

combined_df = pd.concat([mbm_bat, dnb_df], ignore_index=True)

combined_df['batter'] = combined_df.apply(
    lambda row: row['batter'] if pd.notnull(row['batter']) and row['batter'] != ""
    else player_map.get(row['batter_id'], ""), axis=1
)

combined_df['is_DNB'] = combined_df['isOut'].apply(lambda x: 1 if x == "DNB" else 0)
combined_df = combined_df.sort_values(by=['match_id', 'inns', 'is_DNB']).reset_index(drop=True)
combined_df.drop(columns=['is_DNB'], inplace=True)

In [10]:
combined_df['bowler_id'] = combined_df['bowler_id'].fillna("")
combined_df['bowler_id'] = combined_df['bowler'].fillna("")

In [14]:
combined_df[combined_df['match_id']== 548318]

,date,match_id,bat_team,bowl_team,inns,batter_id,batter,runs,balls,fours,sixes,sr,isOut,wicketType,bowler_id,bowler,venue
5709,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,398438,Mayank Agarwal,45,26,5,3,173.08,True,caught,Albie Morkel,Albie Morkel,"MA Chidambaram Stadium, Chepauk"
5710,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,51880,Chris Gayle,68,35,2,6,194.29,True,caught,Ravindra Jadeja,Ravindra Jadeja,"MA Chidambaram Stadium, Chepauk"
5711,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,253802,Virat Kohli,57,46,5,2,123.91,True,caught,Doug Bollinger,Doug Bollinger,"MA Chidambaram Stadium, Chepauk"
5712,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,44936,AB de Villiers,4,2,1,0,200.00,True,caught,Albie Morkel,Albie Morkel,"MA Chidambaram Stadium, Chepauk"
5713,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,35390,Saurabh Tiwary,8,6,1,0,133.33,True,caught,Ravindra Jadeja,Ravindra Jadeja,"MA Chidambaram Stadium, Chepauk"
5714,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,32540,Cheteshwar Pujara,0,1,0,0,0.00,True,bowled,Doug Bollinger,Doug Bollinger,"MA Chidambaram Stadium, Chepauk"
5715,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,38710,Daniel Vettori,1,1,0,0,100.00,False,not out,,NaN,"MA Chidambaram Stadium, Chepauk"
5716,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,27150,Raju Bhatkal,0,2,0,0,0.00,True,caught,Doug Bollinger,Doug Bollinger,"MA Chidambaram Stadium, Chepauk"
5717,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,35731,Vinay Kumar,6,1,0,1,600.00,False,not out,,NaN,"MA Chidambaram Stadium, Chepauk"
5718,2012-04-12,548318,Royal Challengers Bangalore,Chennai Super Kings,1,49636,Muttiah Muralitharan,0,0,0,0,0.00,DNB,DNB,,,"MA Chidambaram Stadium, Chepauk"


In [39]:
combined_df.isnull().sum().to_frame(name="null_values").join(combined_df.notnull().sum().to_frame(name="non_null_values"))


,null_values,non_null_values
date,0,10065
match_id,0,10065
bat_team,0,10065
bowl_team,0,10065
inns,0,10065
batter_id,0,10065
batter,0,10065
runs,0,10065
balls,0,10065
fours,0,10065


In [16]:
combined_df.to_csv(r"D:\ED\FIFS_IPL\data\mbm\cricsheet_ipl_mbm_bat_final.csv", index = False)